# FOML Project — Clean and Prepare Our Custom Image Dataset

This notebook prepares the **real phone photos** and the **AI-generated images** for the first experiment. It does not train a model yet.

It will:
1. Read your two ZIP files without changing the originals.
2. Handle iPhone HEIC/HEIF photos, correct image orientation, and ignore macOS metadata files.
3. Match equivalent category-folder names (for example, `plants` and `plants_and_nature`).
4. check for unreadable files and exact duplicate pixel images.
5. Use the same resizing and JPEG-saving process for both classes.
6. Balance the number of real and AI images within each category.
7. Save a clean dataset ZIP, `metadata.csv`, and a source audit.

**Important:** upload the `real_images.zip` collected by your team and the `ai_generated_100_images.zip` produced by our image-generation notebook. Do not use an unrelated ready-made dataset as a substitute. Keep your original ZIP files unchanged.

## Before running

- In Google Colab, choose **Runtime → Change runtime type**. A GPU is not required for this preprocessing step.
- In the upload dialog, select **both ZIP files** at the same time.
- Our current real-image ZIP has 103 images, but only 16 in plants/nature. Therefore, with the current files, this notebook will create **99 matched images per class**. To get exactly 100 per class as originally planned, take one more real plants/nature photo and add it to the `plants` folder in your ZIP before running this notebook.
- The notebook crops each image to a centered square and resizes it to 256 × 256, using the same method on both classes. Check the preview before using the dataset for training, because a center crop can cut off subjects near the edge.
- We will leave the train/validation/test split for the next stage, after you review this output.

In [ ]:
# Install HEIC/HEIF support. Run this cell first.
!pip -q install pillow-heif

In [ ]:
from google.colab import files
from pathlib import Path

print("Upload BOTH files: real_images.zip and ai_generated_100_images.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
print("\nZIP files uploaded:", zip_names)

# Automatic filename detection. If you renamed a ZIP, edit these two variables manually.
REAL_ZIP = next((n for n in zip_names if "real_images" in n.lower() and "ai" not in n.lower()), None)
AI_ZIP = next((n for n in zip_names if "ai_generated_100_images" in n.lower() or "ai_generated_images" in n.lower()), None)

print("Detected REAL_ZIP:", REAL_ZIP)
print("Detected AI_ZIP:", AI_ZIP)
if REAL_ZIP is None or AI_ZIP is None:
    raise ValueError(
        "I couldn't identify both ZIPs by filename. Set REAL_ZIP and AI_ZIP above to exact names in the uploaded list, then rerun this cell."
    )
if REAL_ZIP == AI_ZIP:
    raise ValueError("The real and AI ZIP names point to the same file. Please upload two different archives.")

In [ ]:
import csv
import hashlib
import json
import random
import re
import shutil
import zipfile
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import PurePosixPath

import pandas as pd
from PIL import Image, ImageOps, ImageFile
from pillow_heif import register_heif_opener

register_heif_opener()
ImageFile.LOAD_TRUNCATED_IMAGES = False

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".heic", ".heif"}
CATEGORY_ORDER = [
    "food",
    "plants_nature",
    "buildings_and_streets",
    "household_objects",
    "vehicles",
    "animals_and_pets",
]
PREFERRED_TARGET_COUNTS = {
    "food": 17,
    "plants_nature": 17,
    "buildings_and_streets": 17,
    "household_objects": 17,
    "vehicles": 16,
    "animals_and_pets": 16,
}

# Folder aliases found in our own collection and in the generated-image notebook.
CATEGORY_ALIASES = {
    "food": "food",
    "plant": "plants_nature",
    "plants": "plants_nature",
    "nature": "plants_nature",
    "plants_nature": "plants_nature",
    "plants_and_nature": "plants_nature",
    "plant_nature": "plants_nature",
    "building_and_street": "buildings_and_streets",
    "buildings_and_streets": "buildings_and_streets",
    "buildings_streets": "buildings_and_streets",
    "streets": "buildings_and_streets",
    "household": "household_objects",
    "household_object": "household_objects",
    "household_objects": "household_objects",
    "vehicle": "vehicles",
    "vehicles": "vehicles",
    "animal": "animals_and_pets",
    "animals": "animals_and_pets",
    "pets": "animals_and_pets",
    "animals_pets": "animals_and_pets",
    "animals_and_pets": "animals_and_pets",
}


def normalize_folder_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def category_from_member(member_name):
    path = PurePosixPath(member_name)
    for part in reversed(path.parts[:-1]):
        cleaned = normalize_folder_name(part)
        if cleaned in CATEGORY_ALIASES:
            return CATEGORY_ALIASES[cleaned]
    return None


def open_rgb_oriented(raw_bytes):
    with Image.open(BytesIO(raw_bytes)) as im:
        im = ImageOps.exif_transpose(im)
        return im.convert("RGB")


def image_pixel_hash(image):
    # Hash decoded, orientation-corrected RGB pixels to catch exact image duplicates
    # even when the files have different names or metadata.
    h = hashlib.sha256()
    h.update(str(image.size).encode("utf-8"))
    h.update(image.tobytes())
    return h.hexdigest()


def scan_archive(zip_path, label):
    candidates = []
    errors = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            member = info.filename
            parts = PurePosixPath(member).parts
            # Ignore macOS resource-fork files and common hidden system files.
            if any(part == "__MACOSX" or part.startswith("._") or part == ".DS_Store" for part in parts):
                continue
            if PurePosixPath(member).suffix.lower() not in IMAGE_EXTS:
                continue
            category = category_from_member(member)
            if category is None:
                errors.append({"label": label, "member": member, "reason": "No recognized category folder"})
                continue
            try:
                raw = zf.read(info)
                image = open_rgb_oriented(raw)
                width, height = image.size
                if width < 16 or height < 16:
                    raise ValueError(f"Image dimensions too small: {width}x{height}")
                pixel_hash = image_pixel_hash(image)
                candidates.append({
                    "label": label,
                    "category": category,
                    "source_zip": str(zip_path),
                    "member": member,
                    "extension": PurePosixPath(member).suffix.lower(),
                    "original_width": width,
                    "original_height": height,
                    "source_file_sha256": hashlib.sha256(raw).hexdigest(),
                    "pixel_sha256": pixel_hash,
                    "status": "readable_candidate",
                    "skip_reason": "",
                })
            except Exception as e:
                errors.append({"label": label, "member": member, "reason": f"Could not decode: {type(e).__name__}: {e}"})
    return candidates, errors

print("Helper functions loaded. Next, scan both ZIP files.")

In [ ]:
real_candidates, real_errors = scan_archive(REAL_ZIP, "real")
ai_candidates, ai_errors = scan_archive(AI_ZIP, "ai_generated")
scan_errors = real_errors + ai_errors

print(f"Readable real-image candidates: {len(real_candidates)}")
print(f"Readable AI-image candidates:   {len(ai_candidates)}")
print(f"Files skipped or needing attention: {len(scan_errors)}")

if scan_errors:
    display(pd.DataFrame(scan_errors).head(30))


def category_count_table(candidates):
    counts = Counter(r["category"] for r in candidates)
    return pd.DataFrame([
        {"category": category, "count": counts.get(category, 0)}
        for category in CATEGORY_ORDER
    ])

print("\nREAL category counts:")
display(category_count_table(real_candidates))
print("AI-GENERATED category counts:")
display(category_count_table(ai_candidates))

## Select a balanced, matched pilot set

The notebook uses the same maximum number of images from each subject category for both labels. If a category is short in either archive, the output count for that category is reduced rather than accidentally giving one class more examples of that subject.

In [ ]:
# Remove exact duplicate pixel images within each class.
def drop_exact_pixel_duplicates(candidates):
    seen = {}
    unique = []
    duplicate_rows = []
    for record in sorted(candidates, key=lambda r: (r["category"], r["member"])):
        key = (record["category"], record["pixel_sha256"])
        if key in seen:
            record["status"] = "excluded_duplicate_pixels"
            record["skip_reason"] = f"Same decoded pixels as {seen[key]}"
            duplicate_rows.append(record.copy())
        else:
            seen[key] = record["member"]
            unique.append(record)
    return unique, duplicate_rows

real_unique, real_duplicates = drop_exact_pixel_duplicates(real_candidates)
ai_unique, ai_duplicates = drop_exact_pixel_duplicates(ai_candidates)

# If identical pixel content appears in both classes, stop rather than train on contradictory labels.
real_hashes = {r["pixel_sha256"]: r["member"] for r in real_unique}
ai_hashes = {r["pixel_sha256"]: r["member"] for r in ai_unique}
cross_class_duplicate_hashes = set(real_hashes) & set(ai_hashes)
if cross_class_duplicate_hashes:
    examples = [
        {"real_member": real_hashes[h], "ai_member": ai_hashes[h]}
        for h in list(cross_class_duplicate_hashes)[:10]
    ]
    display(pd.DataFrame(examples))
    raise ValueError("Identical decoded pixels were found in both labels. Inspect the files and fix the labels before continuing.")

real_counts = Counter(r["category"] for r in real_unique)
ai_counts = Counter(r["category"] for r in ai_unique)
TARGET_COUNTS = {
    category: min(PREFERRED_TARGET_COUNTS[category], real_counts.get(category, 0), ai_counts.get(category, 0))
    for category in CATEGORY_ORDER
}

missing = [category for category, count in TARGET_COUNTS.items() if count == 0]
if missing:
    raise ValueError(f"No matched images available for these categories: {missing}. Check folder names and ZIP files.")

total_per_class = sum(TARGET_COUNTS.values())
print("Matched counts we can use:")
display(pd.DataFrame([{"category": c, "target_per_class": TARGET_COUNTS[c], "real_available": real_counts.get(c, 0), "ai_available": ai_counts.get(c, 0)} for c in CATEGORY_ORDER]))
print(f"Total per class: {total_per_class}; total images after processing: {total_per_class * 2}")
if total_per_class < 100:
    print("NOTE: The current real ZIP has only 16 plants/nature images, so this run will produce 99 real + 99 AI images. Add one more real plants/nature photo to reach the planned 100 per class.")

SELECTION_SEED = 42
rng = random.Random(SELECTION_SEED)
selected = []
for label, records in [("real", real_unique), ("ai_generated", ai_unique)]:
    for category in CATEGORY_ORDER:
        candidates = sorted([r for r in records if r["category"] == category], key=lambda r: r["member"])
        chosen = rng.sample(candidates, TARGET_COUNTS[category])
        for record in chosen:
            copied = record.copy()
            copied["status"] = "selected"
            copied["skip_reason"] = ""
            selected.append(copied)

assert Counter(r["label"] for r in selected) == {"real": total_per_class, "ai_generated": total_per_class}
print("Selection complete; both classes have matching category counts.")

## Standardize the selected images

We will save new **256 × 256 RGB JPEG** copies, with corrected orientation and without carrying over the original EXIF metadata. The originals remain in your ZIP files. Both classes use the exact same crop and export settings.

In [ ]:
OUTPUT_DIR = Path("processed_foml_dataset")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (256, 256)
JPEG_QUALITY = 95

# Open each input ZIP once for more efficient extraction.
zip_handles = {
    REAL_ZIP: zipfile.ZipFile(REAL_ZIP, "r"),
    AI_ZIP: zipfile.ZipFile(AI_ZIP, "r"),
}
metadata_rows = []
standardized_hashes = {}

try:
    per_category_counters = defaultdict(int)
    for record in selected:
        with zip_handles[record["source_zip"]].open(record["member"], "r") as f:
            raw = f.read()
        image = open_rgb_oriented(raw)
        # Same centered square crop and resize for every image, regardless of label.
        image = ImageOps.fit(
            image,
            IMAGE_SIZE,
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5),
        ).convert("RGB")
        normalized_hash = hashlib.sha256(image.tobytes()).hexdigest()
        if normalized_hash in standardized_hashes:
            previous = standardized_hashes[normalized_hash]
            raise ValueError(
                "Two selected images became pixel-identical after preprocessing: "
                f"{previous} and {record['member']}. Inspect them before training."
            )
        standardized_hashes[normalized_hash] = record["member"]

        per_category_counters[(record["label"], record["category"])] += 1
        number = per_category_counters[(record["label"], record["category"])]
        label_prefix = "REAL" if record["label"] == "real" else "AI"
        output_name = f"{label_prefix}_{record['category']}_{number:03d}.jpg"
        output_path = OUTPUT_DIR / record["label"] / record["category"] / output_name
        output_path.parent.mkdir(parents=True, exist_ok=True)
        image.save(output_path, format="JPEG", quality=JPEG_QUALITY, optimize=True, exif=b"")

        metadata_rows.append({
            "image_id": f"{label_prefix}_{record['category']}_{number:03d}",
            "relative_path": output_path.as_posix(),
            "label": record["label"],
            "category": record["category"],
            "source_zip": Path(record["source_zip"]).name,
            "source_member": record["member"],
            "original_extension": record["extension"],
            "original_width": record["original_width"],
            "original_height": record["original_height"],
            "processed_width": IMAGE_SIZE[0],
            "processed_height": IMAGE_SIZE[1],
            "processed_format": "JPEG",
            "processed_jpeg_quality": JPEG_QUALITY,
            "source_file_sha256": record["source_file_sha256"],
            "source_pixel_sha256": record["pixel_sha256"],
            "preprocessing": "EXIF transpose; RGB; centered square crop; resize 256x256; JPEG quality 95; metadata stripped",
            "split": "pending",
        })
finally:
    for zf in zip_handles.values():
        zf.close()

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(OUTPUT_DIR / "metadata.csv", index=False, encoding="utf-8-sig")

readme = f"""FOML AI-generated vs Real Image Detection — processed pilot dataset

Images: {len(metadata_rows)}
Real images: {sum(1 for r in metadata_rows if r['label'] == 'real')}
AI-generated images: {sum(1 for r in metadata_rows if r['label'] == 'ai_generated')}
Image size: {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}
Format: RGB JPEG, quality {JPEG_QUALITY}

All images were processed using the same orientation correction, centered square crop,
resize, RGB conversion, and JPEG settings. Original archive paths and file hashes are in metadata.csv.
The split column is intentionally 'pending': create train/validation/test splits only after reviewing this dataset.

Limitation: if all AI examples come from one generator, this pilot cannot establish that a detector
will generalize to images from other generators. Report this limitation in the FOML project.
"""
(OUTPUT_DIR / "README.txt").write_text(readme, encoding="utf-8")

print("Saved processed dataset to:", OUTPUT_DIR.resolve())
print("Saved metadata to:", (OUTPUT_DIR / "metadata.csv").resolve())
display(metadata_df.groupby(["label", "category"]).size().unstack(fill_value=0))

## Review the output and create a ZIP download

Review the category-count table and sample preview. If crops removed important subjects or the labels/folders look wrong, stop and correct that before training.

In [ ]:
# Preview 2 real and 2 AI images for each category.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(CATEGORY_ORDER), 4, figsize=(12, 3 * len(CATEGORY_ORDER)))
for row_idx, category in enumerate(CATEGORY_ORDER):
    for col_idx, label in enumerate(["real", "real", "ai_generated", "ai_generated"]):
        folder = OUTPUT_DIR / label / category
        files_in_folder = sorted(folder.glob("*.jpg"))
        same_label_offset = col_idx if label == "real" else col_idx - 2
        ax = axes[row_idx, col_idx]
        if same_label_offset < len(files_in_folder):
            with Image.open(files_in_folder[same_label_offset]) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(f"{label} — {category}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

# Validate every saved image and confirm dimensions.
output_images = list(OUTPUT_DIR.rglob("*.jpg"))
assert len(output_images) == len(metadata_df), "Output image count does not match metadata rows."
for image_path in output_images:
    with Image.open(image_path) as im:
        assert im.size == IMAGE_SIZE, f"Unexpected size in {image_path}: {im.size}"
        assert im.mode == "RGB", f"Unexpected color mode in {image_path}: {im.mode}"

print(f"Validated {len(output_images)} processed images: all are RGB JPEGs at {IMAGE_SIZE[0]}×{IMAGE_SIZE[1]}.")

In [ ]:
# Write a simple audit file with the source images that were selected or omitted.
selected_keys = {(r["label"], r["member"]) for r in selected}
audit_rows = []
for r in real_candidates + ai_candidates:
    key = (r["label"], r["member"])
    if key in selected_keys:
        status, reason = "selected_and_processed", ""
    elif r.get("status") == "excluded_duplicate_pixels":
        status, reason = "omitted_duplicate_pixels", r.get("skip_reason", "")
    else:
        status, reason = "omitted_to_match_category_counts", "Not selected for balanced pilot quota"
    audit_rows.append({
        "label": r["label"],
        "category": r["category"],
        "source_zip": Path(r["source_zip"]).name,
        "source_member": r["member"],
        "status": status,
        "reason": reason,
        "original_width": r["original_width"],
        "original_height": r["original_height"],
    })
for e in scan_errors:
    audit_rows.append({
        "label": e["label"], "category": "", "source_zip": Path(REAL_ZIP if e["label"] == "real" else AI_ZIP).name,
        "source_member": e["member"], "status": "skipped_during_scan", "reason": e["reason"],
        "original_width": "", "original_height": "",
    })
pd.DataFrame(audit_rows).to_csv(OUTPUT_DIR / "source_audit.csv", index=False, encoding="utf-8-sig")

# Package the processed images plus both CSVs and README.
OUTPUT_ZIP = Path("processed_foml_dataset.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent).as_posix())

print(f"ZIP ready: {OUTPUT_ZIP.resolve()} ({OUTPUT_ZIP.stat().st_size / (1024 * 1024):.1f} MB)")
print("Contains: real/ and ai_generated/ image folders, metadata.csv, source_audit.csv, README.txt")
files.download(str(OUTPUT_ZIP))

## What to do after this notebook

1. Download and keep `processed_foml_dataset.zip` safely.
2. Verify that both classes have the same number of images in each category. Check `metadata.csv` and `source_audit.csv`.
3. Do not evaluate a model on the same images used for training. The next notebook should create a reproducible train/validation/test split and train a baseline CNN.
4. Treat this as a small pilot. To make the detector more credible, collect a larger set and, if possible, test against an AI generator not used to create the training images.